In [ ]:
import pandas as pd

disease = pd.read_csv("../data/training_data.csv")
reviews = pd.concat([
    pd.read_csv("../data/drugsComTrain_raw.csv"),
    pd.read_csv("../data/drugsComTest_raw.csv"),
])

print("Disease data:", disease.shape)
print("Reviews:", reviews.shape)

Disease data: (4920, 134)
Reviews: (215063, 7)


In [ ]:
# 1. Drop the empty extra column
disease = disease.drop(columns=["Unnamed: 133"])

# 2. Tidy column names: remove stray spaces
disease.columns = disease.columns.str.strip().str.replace(" ", "")

# 3. Check the result
print("Shape:", disease.shape)
print("Missing values:", disease.isna().sum().sum())
print("Number of diseases:", disease["prognosis"].nunique())
print(disease["prognosis"].value_counts().head())

Shape: (4920, 133)
Missing values: 0
Number of diseases: 41
prognosis
Fungal infection       120
Allergy                120
GERD                   120
Chronic cholestasis    120
Drug Reaction          120
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Inputs and answer
X = disease.drop(columns=["prognosis"])
y = disease["prognosis"]

# 2. Split 80% / 20%, keeping all 41 diseases balanced in both parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Train the model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 4. Test it
pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))

Accuracy: 1.0


In [ ]:
# 1. Duplicates in the whole dataset
print("Duplicate rows:", disease.duplicated().sum())
print("Unique rows:", len(disease.drop_duplicates()))

# 2. How many test rows are exact copies of training rows
train_rows = set(map(tuple, X_train.values))
leak = sum(tuple(r) in train_rows for r in X_test.values)
print("Test rows also in training:", leak, "of", len(X_test))

Duplicate rows: 4616
Unique rows: 304
Test rows also in training: 984 of 984


In [ ]:
# 1. Keep only unique rows
unique = disease.drop_duplicates()
print("Unique rows:", unique.shape[0])

# 2. Does the same symptom pattern ever map to two different diseases?
symptoms_only = unique.drop(columns=["prognosis"])
print("Conflicting patterns:", symptoms_only.duplicated().sum())

# 3. Split and retrain on unique rows only
Xu = unique.drop(columns=["prognosis"])
yu = unique["prognosis"]

Xu_train, Xu_test, yu_train, yu_test = train_test_split(
    Xu, yu, test_size=0.2, random_state=42, stratify=yu
)

model_fair = RandomForestClassifier(n_estimators=100, random_state=42)
model_fair.fit(Xu_train, yu_train)

print("Fair accuracy:", accuracy_score(yu_test, model_fair.predict(Xu_test)))

Unique rows: 304
Conflicting patterns: 0
Fair accuracy: 1.0


In [ ]:
import numpy as np

rng = np.random.default_rng(0)

for k in [1, 2, 3, 4, 5]:
    correct, total = 0, 0
    for row, true_disease in zip(Xu_test.values, yu_test.values):
        present = np.flatnonzero(row)          # positions of symptoms that are 1
        if len(present) < k:
            continue
        keep = rng.choice(present, k, replace=False)
        partial = np.zeros(len(row))
        partial[keep] = 1                      # keep only k symptoms
        guess = model_fair.predict(pd.DataFrame([partial], columns=Xu.columns))[0]
        correct += (guess == true_disease)
        total += 1
    print(f"{k} symptom(s): accuracy {correct/total:.2f}  ({total} cases)")

1 symptom(s): accuracy 0.39  (61 cases)
2 symptom(s): accuracy 0.56  (61 cases)
3 symptom(s): accuracy 0.64  (61 cases)
4 symptom(s): accuracy 0.73  (56 cases)
5 symptom(s): accuracy 0.86  (49 cases)


In [ ]:
rng = np.random.default_rng(0)
classes = model_fair.classes_

for k in [1, 2, 3, 4, 5]:
    top1, top3, total = 0, 0, 0
    for row, true_disease in zip(Xu_test.values, yu_test.values):
        present = np.flatnonzero(row)
        if len(present) < k:
            continue
        keep = rng.choice(present, k, replace=False)
        partial = np.zeros(len(row))
        partial[keep] = 1

        probs = model_fair.predict_proba(pd.DataFrame([partial], columns=Xu.columns))[0]
        best3 = classes[np.argsort(probs)[-3:]]      # names of the 3 highest
        top1 += (classes[np.argmax(probs)] == true_disease)
        top3 += (true_disease in best3)
        total += 1
    print(f"{k} symptom(s): top-1 {top1/total:.2f} | top-3 {top3/total:.2f}")
    

1 symptom(s): top-1 0.39 | top-3 0.51
2 symptom(s): top-1 0.56 | top-3 0.74
3 symptom(s): top-1 0.64 | top-3 0.85
4 symptom(s): top-1 0.73 | top-3 0.91
5 symptom(s): top-1 0.86 | top-3 0.96


In [ ]:
import joblib, os

os.makedirs("../models", exist_ok=True)

joblib.dump(model_fair, "../models/disease_model.joblib")
joblib.dump(list(Xu.columns), "../models/symptom_list.joblib")

print("Saved. Symptoms in list:", len(Xu.columns))
print(os.listdir("../models"))

Saved. Symptoms in list: 132
['disease_model.joblib', 'symptom_list.joblib']


In [ ]:
print("Before:", reviews.shape)

# 1. Remove rows with no condition
reviews = reviews.dropna(subset=["condition"])

# 2. Remove rows where the condition is leftover web-page text
reviews = reviews[~reviews["condition"].str.contains("</span>")]

# 3. Trim spaces and rebuild the row numbers
reviews["condition"] = reviews["condition"].str.strip()
reviews = reviews.reset_index(drop=True)

print("After:", reviews.shape)
print("Different conditions:", reviews["condition"].nunique())
print(reviews["condition"].value_counts().head(5))

Before: (215063, 7)
After: (212698, 7)
Different conditions: 836
condition
Birth Control    38436
Depression       12164
Pain              8245
Anxiety           7812
Acne              7435
Name: count, dtype: int64


In [ ]:
mapping = {
    "Acne": ["Acne"],
    "Allergy": ["Allergies"],
    "Bronchial Asthma": ["Asthma", "Asthma, Maintenance", "Asthma, acute"],
    "Common Cold": ["Cold Symptoms"],
    "Diabetes": ["Diabetes, Type 2", "Diabetes, Type 1"],
    "Dimorphic hemmorhoids(piles)": ["Hemorrhoids"],
    "GERD": ["GERD"],
    "Gastroenteritis": ["Gastroenteritis"],
    "Heart attack": ["Heart Attack"],
    "Hepatitis B": ["Hepatitis B"],
    "Hepatitis C": ["Hepatitis C"],
    "Hypertension": ["High Blood Pressure"],
    "Hyperthyroidism": ["Hyperthyroidism"],
    "Hypoglycemia": ["Hypoglycemia"],
    "Hypothyroidism": ["Underactive Thyroid", "Hypothyroidism, After Thyroid Removal"],
    "Malaria": ["Malaria"],
    "Migraine": ["Migraine"],
    "Pneumonia": ["Pneumonia"],
    "Psoriasis": ["Psoriasis", "Plaque Psoriasis"],
    "Urinary tract infection": ["Urinary Tract Infection"],
    "Varicose veins": ["Varicose Veins"],
}

# Build a lookup: review condition -> disease
lookup = {cond: dis for dis, conds in mapping.items() for cond in conds}
reviews["disease"] = reviews["condition"].map(lookup)

# Reviews per disease
print(reviews["disease"].value_counts().to_string())

# Diseases from the model that have no reviews
all_diseases = [d.strip() for d in disease["prognosis"].unique()]
missing = [d for d in all_diseases if d not in mapping]
print("\nNo review data for", len(missing), "diseases:")
print(missing)

disease
Acne                            7435
Diabetes                        3547
Hypertension                    3104
Migraine                        2277
Urinary tract infection         1747
Psoriasis                       1226
Bronchial Asthma                1130
GERD                             968
Hepatitis C                      901
Hypothyroidism                   755
Pneumonia                        364
Allergy                          228
Common Cold                      151
Dimorphic hemmorhoids(piles)     118
Heart attack                      74
Gastroenteritis                   48
Malaria                           26
Hepatitis B                       20
Hyperthyroidism                   15
Hypoglycemia                       4
Varicose veins                     3

No review data for 20 diseases:
['Fungal infection', 'Chronic cholestasis', 'Drug Reaction', 'Peptic ulcer diseae', 'AIDS', 'Cervical spondylosis', 'Paralysis (brain hemorrhage)', 'Jaundice', 'Chicken pox', 'Dengue

In [ ]:
rated = reviews.dropna(subset=["disease"])

def top_medicines(disease_name, n=5, m=10, min_reviews=3):
    sub = rated[rated["disease"] == disease_name]
    if sub.empty:
        return None

    disease_avg = sub["rating"].mean()
    g = sub.groupby("drugName").agg(
        reviews=("rating", "size"),
        avg_rating=("rating", "mean"),
    ).reset_index()

    g = g[g["reviews"] >= min_reviews]
    if g.empty:
        return None

    g["score"] = (g["reviews"] * g["avg_rating"] + m * disease_avg) / (g["reviews"] + m)
    return g.sort_values("score", ascending=False).head(n).round(2)

print(top_medicines("Hypertension"))
print()
print(top_medicines("Migraine"))

                             drugName  reviews  avg_rating  score
13            Amlodipine / olmesartan       53        8.62   8.24
23                               Azor       53        8.62   8.24
27                            Benicar       80        7.64   7.48
110                        Olmesartan       94        7.54   7.41
67   Hydrochlorothiazide / irbesartan        6        9.33   7.37

                  drugName  reviews  avg_rating  score
24                   Frova       36        9.36   9.08
25            Frovatriptan       39        9.33   9.08
49             Rizatriptan      204        8.95   8.91
53                Treximet       82        9.01   8.91
40  Naproxen / sumatriptan       89        8.99   8.90


In [ ]:
import joblib

model = joblib.load("../models/disease_model.joblib")
symptom_list = joblib.load("../models/symptom_list.joblib")

def recommend(symptoms, top_k=3):
    unknown = [s for s in symptoms if s not in symptom_list]
    if unknown:
        print("Unknown symptoms:", unknown)
        return

    # 1 = symptom present, 0 = absent, in the order the model expects
    x = pd.DataFrame([[1 if s in symptoms else 0 for s in symptom_list]],
                     columns=symptom_list)
    probs = model.predict_proba(x)[0]

    for i in np.argsort(probs)[::-1][:top_k]:
        name = model.classes_[i].strip()
        print(f"\n{name}  ({probs[i]*100:.0f}%)")
        meds = top_medicines(name, n=3)
        if meds is None:
            print("   No review data available for this disease.")
        else:
            print(meds[["drugName", "reviews", "avg_rating", "score"]].to_string(index=False))

recommend(["continuous_sneezing", "chills", "fatigue", "cough", "high_fever"])


Bronchial Asthma  (35%)
    drugName  reviews  avg_rating  score
Levalbuterol       12        9.58   8.36
     Xopenex        7        9.57   7.99
 Epinephrine        7        9.57   7.99

Allergy  (22%)
      drugName  reviews  avg_rating  score
   Hydroxyzine       28        9.04   8.21
        Atarax       11        9.00   7.53
Levocetirizine       97        5.28   5.34

Pneumonia  (8%)
    drugName  reviews  avg_rating  score
    Rocephin       14        8.93   7.99
 Ceftriaxone       14        8.93   7.99
Moxifloxacin       37        8.30   7.95


In [ ]:
frames = []
for d in mapping:
    t = top_medicines(d, n=1000)       # n=1000 means "keep every drug"
    if t is not None:
        t.insert(0, "disease", d)
        frames.append(t)

scores = pd.concat(frames, ignore_index=True)
scores.to_csv("../models/medicine_scores.csv", index=False)

print("Saved table:", scores.shape)
print(scores.groupby("disease").size().sort_values(ascending=False).to_string())

Saved table: (571, 5)
disease
Hypertension                    105
Acne                             83
Diabetes                         77
Bronchial Asthma                 55
Migraine                         51
Psoriasis                        44
GERD                             29
Urinary tract infection          26
Common Cold                      19
Hepatitis C                      19
Pneumonia                        16
Hypothyroidism                   13
Dimorphic hemmorhoids(piles)     12
Heart attack                      7
Malaria                           6
Allergy                           4
Gastroenteritis                   2
Hepatitis B                       2
Hyperthyroidism                   1


In [ ]:
# 1. Add the two mappings
mapping["Osteoarthristis"] = ["Osteoarthritis"]
mapping["Arthritis"] = ["Rheumatoid Arthritis"]

# 2. Re-tag the reviews with the updated mapping
lookup = {cond: dis for dis, conds in mapping.items() for cond in conds}
reviews["disease"] = reviews["condition"].map(lookup)
rated = reviews.dropna(subset=["disease"])

# 3. Rebuild and save the medicine scores
frames = []
for d in mapping:
    t = top_medicines(d, n=1000)
    if t is not None:
        t.insert(0, "disease", d)
        frames.append(t)

scores = pd.concat(frames, ignore_index=True)
scores.to_csv("../models/medicine_scores.csv", index=False)

print("Saved table:", scores.shape)
print(scores.groupby("disease").size()[["Osteoarthristis", "Arthritis"]])

Saved table: (685, 5)
disease
Osteoarthristis    56
Arthritis          58
dtype: int64
